<a href="https://colab.research.google.com/github/AdelineKwakye/ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026-10-16%20%E2%80%94%20Walmart%20Warm-Up%20SKU%20Mess%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '/content/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# TODO (hint: .str.lower().str.contains('pop'))
cat_containingpop = cat[cat['product_name'].str.lower().str.contains('pop', na=False)] # filter on matches with pop

print("Number of rows containg pop, ", len(cat_containingpop))
display(cat_containingpop)


Number of rows containg pop,  4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
# TODO

cat_contaningpop = cat[cat['product_name'].str.lower().str.contains('pop', na=False)]
display(cat_containingpop) # pop tarts
print("number of poptarts: ", len(cat_contaningpop))
cat_notstrawberry_poptarts = cat[
    cat['product_name'].str.lower().str.contains('pop', na=False)
    & ~cat['product_name'].str.lower().str.contains('strawberry', na=False)
]
display(cat_notstrawberry_poptarts)
print("number of non strawberry poptarts ", len(cat_notstrawberry_poptarts))


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


number of poptarts:  4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size


number of non strawberry poptarts  0


**Pop-Tart rows that are a different product:**  No rows were returned by the filter that didn't contain "strawberry". There were no non-strawberry Pop-Tart products. There were four rows that represented the same strawberry Pop-Tart produce despite having different SKUs.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
# TODO
CANON = {
    'PT-12' : 'POPTART-STRAW',
    'SKU#459812' : 'POPTART-STRAW',
    'POPTART-STRAW': 'POPTART-STRAW',
    'PT_STRAWBERRY' : 'POPTART-STRAW'
}

def consolidate(df):
    out = df.copy()
    out['canonical_sku'] = out['sku'].map(CANON).fillna(out['sku'])
    return out
out = consolidate(cat)
print("consolidated groups\n", out.groupby('canonical_sku').size().sort_values(ascending=False))

consolidated groups
 canonical_sku
POPTART-STRAW    4
BABYFORMULA      1
BATT-100         1
BATT_AA_4PK      1
BEER-24          1
BFORM-01         1
BG-MONO          1
BREAD-WH         1
CANDLE-EMRG      1
BAT-AA4          1
CARDS-01         1
CHIPS-VP         1
CHRG-PORT        1
CHIPVARIETY      1
CNSOUP-01        1
DIAP-S3          1
DIAPERS-3        1
CNDL-EM          1
FIRSTAID-KIT     1
FLASH-01         1
GAME-MONO        1
HDTARP-01        1
LED-FLASH        1
LTBEER24         1
MONOPOLY-01      1
FAID-01          1
PAPERTOWEL6      1
PB-16            1
PLAYCARD-STD     1
PNUTBTR-16       1
PORTCHARGER      1
PTOWEL-6         1
ROKU-STD         1
SOUP-CAN         1
STREAM-01        1
TARP-HD          1
TOILETPAPER12    1
TP-12            1
WATER24PK        1
WHITEBREAD       1
WTR-24           1
WTR_24PK         1
dtype: int64


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
# TODO
before = out['category']
print("distinct category values before, ", before.value_counts(dropna=False))
print("number of distinct categories: , ", before.nunique())
out['category_clean'] = out['category'].str.strip().str.title()

CATEGORY_CANON = {
    'Emergency' : 'Emergency',
    'Emergency Supplies' : 'Emergency',
    'Breakfast' : 'Breakfast',
    'Breakfast Foods' : 'Breakfast',
    'Alcohol' : 'Alcohol',
    'Alcoholic Beverages' : 'Alcohol',
    'Grocery' : 'Grocery',
    'Groceries' : 'Grocery',
    'Baby' : 'Baby',
    'Baby Products' : 'Baby'
}
out['category_clean'] = out['category'].map(CATEGORY_CANON).fillna(out['category'])
print("distinct category values after", out['category_clean'].value_counts(dropna=False))
print("number of distinct categories after: ", out['category_clean'].nunique())


distinct category values before,  category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64
number of distinct categories: ,  27
distinct category values after category_clean
Emergency        8
Breakfast        3
EMERGENCY        3
ENTERTAINMENT    3
GROCERY          3
Baby             3
health           2
ELECTRONICS      2
Grocery          2
ho

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
# TODO
def normalize_name(name):
  name = name.lower().strip()
  name = name.replace("-"," ").replace("_", " ").replace(".", " ").replace("'"," ").replace("(","").replace(")","")
  words = name.split() # separate the names into individual words
  return " ".join(sorted(words)) # joins words together into one string

def normalize_product_name(df):
  out = df.copy()
  out['normalized_product_name'] = out['product_name'].apply(normalize_name)
  return out

normalized = normalize_product_name(out)
print("\nnormalized product names: \n")
display(normalized[['product_name', 'normalized_product_name']].head(5))

groupby_sku = normalized.groupby('normalized_product_name')['sku'].nunique()
print("\n normalized names grouped by sku: \n")
display(groupby_sku)

fragmented_names = groupby_sku[groupby_sku > 1]
print("\n fragmented names: \n")
display(fragmented_names)

candidates = normalized[normalized['normalized_product_name'].isin(fragmented_names.index)].copy()
print("\ncandidate list: \n")
display(candidates[
    ['normalized_product_name', 'sku', 'product_name']
].sort_values(['normalized_product_name', 'sku']))



normalized product names: 



,product_name,normalized_product_name
0,pop-tarts strawberry,pop strawberry tarts
1,Strawberry Pop-Tart,pop strawberry tart
2,Pop-Tarts Strawberry,pop strawberry tarts
3,pop-tarts strawberry,pop strawberry tarts
4,BOTTLED WATER 24PK,24pk bottled water



 normalized names grouped by sku: 



,sku
normalized_product_name,
12 paper roll toilet,1
12pk paper toilet,1
24 beer light pack,2
24 bottled pack water,1
24 pack water,1
24pk bottled water,1
3 baby diapers size,1
3 diapers sz,1
4 aa batteries pack,2



 fragmented names: 



,sku
normalized_product_name,
24 beer light pack,2
4 aa batteries pack,2
aid first kit,2
board game monopoly,3
bread white,2
butter peanut,2
candles emergency,2
canned soup,2
cards playing,2



candidate list: 



,normalized_product_name,sku,product_name
12,24 beer light pack,BEER-24,Light Beer (24-pack)
13,24 beer light pack,LTBEER24,Light Beer (24-pack)
8,4 aa batteries pack,BAT-AA4,AA Batteries 4-Pack
9,4 aa batteries pack,BATT_AA_4PK,AA Batteries 4-Pack
29,aid first kit,FAID-01,First-Aid Kit
30,aid first kit,FIRSTAID-KIT,First Aid Kit
14,board game monopoly,BG-MONO,Monopoly Board Game
16,board game monopoly,GAME-MONO,Board Game - Monopoly
15,board game monopoly,MONOPOLY-01,Monopoly Board Game
23,bread white,BREAD-WH,WHITE BREAD


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
# TODO
assert out['canonical_sku'].notna().all(), 'canonical_sku contains null values'
assert len(out) == len(cat), 'consolidation changed the number of rows'

print("Both assertions passed!")

Both assertions passed!


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) Two product families that also appear to be fragmented would be the 24 beer pack and AA batteries. The 24 beer pack appear under the SKUs as BEER-24 and LTBEER24, but both product names appear as "Light Beer (24-pack)", so I would group them under one canonical SKU. I would do the same thing with the AA batteries. They appear under the SKUs BAT-AA4 and BATT_AA_4PK, but both product names are "AA Batteries 4-Pack" so I would also consolidated them into one family.

b) Consolidation becomes a business decision when you can't definitively tell if two entries are representing the same exact product based on their product names. From this catalog we see that PTOWEL-6 and PAPERTOWEL6 are both listed as "Paper Towels". Although they both appear to be the same product, I'd want to confirm that they both have the same amount of rolls, size, and brand before combining them. I would ask whoever is in charge of inventory to verify the product specifications prior to making the change.